# 성별_연령 및 소득 구간 추가 실험

- 실험명: income_sex_age_experiment
- 모델: Multinomial Logistic, HistGradientBoosting
- 순위 가중치: 1순위 1.5, 2순위 1.25, 3순위 1.0
- 소득 변수: 기준 중위소득 50% 기준 이진 구간, 기준 중위소득 비율 다층 구간


## 1. 패키지 및 경로

- 결과는 노트북 출력으로만 확인함.
- 별도 CSV 파일은 생성하지 않음.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

BASE_PATH = Path.cwd()
while BASE_PATH.name != "oracle_mnc_project" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

RAW_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "source" / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"
MAPPING_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "processed" / "satisfaction" / "ml_activity_category_mapping.csv"

print("RAW_PATH 존재:", RAW_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())


## 2. 소득 구간 설계

- 설문 가구소득은 구간형 코드임.
- 가구원 수와 조사년도를 이용해 기준 중위소득 대비 비율을 근사함.
- 소득구간 이진: 중위소득 50% 이하 / 초과
- 소득구간 다층: 50% 이하 / 50~100% / 100~150% / 150% 초과


In [ ]:
raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")
mapping.columns = ["activity_code", "activity_name", "category", "use_target"]

# 기준 중위소득 100%, 원/월
median_income_100 = {
    2024: {
        1: 2228445, 2: 3682609, 3: 4714657, 4: 5729913,
        5: 6695735, 6: 7618369, 7: 8514994,
    },
    2025: {
        1: 2392013, 2: 3932658, 3: 5025353, 4: 6097773,
        5: 7108192, 6: 8064805, 7: 8988428,
    },
}

# 설문 가구소득 코드의 월 소득 구간, 만원 단위
# 1: 100만원 미만, 2: 100~200만원, ..., 7: 600만원 이상
income_mid_10k = {
    1: 50,
    2: 150,
    3: 250,
    4: 350,
    5: 450,
    6: 550,
    7: 650,
}

def household_size_cap(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x < 1:
        return np.nan
    return min(x, 7)

def median_ratio(row):
    year = int(row["조사년도"])
    size = household_size_cap(row["동거가구원수_생성"])
    income_code = row["가구소득_생성"]

    if pd.isna(size) or pd.isna(income_code):
        return np.nan

    income_won = income_mid_10k.get(int(income_code), np.nan) * 10000
    median_won = median_income_100.get(year, {}).get(size, np.nan)

    if pd.isna(income_won) or pd.isna(median_won) or median_won == 0:
        return np.nan

    return income_won / median_won

raw["중위소득비율_근사"] = raw.apply(median_ratio, axis=1)

raw["소득구간_50이진"] = np.where(
    raw["중위소득비율_근사"].isna(),
    "unknown",
    np.where(raw["중위소득비율_근사"] <= 0.5, "50이하", "50초과")
)

raw["소득구간_다층"] = pd.cut(
    raw["중위소득비율_근사"],
    bins=[-np.inf, 0.5, 1.0, 1.5, np.inf],
    labels=["50이하", "50_100", "100_150", "150초과"]
).astype(str)
raw.loc[raw["중위소득비율_근사"].isna(), "소득구간_다층"] = "unknown"

print("2024~2025 소득구간 이진 분포")
print(raw.loc[raw["조사년도"].isin([2024, 2025]), "소득구간_50이진"].value_counts(dropna=False))

print("\n2024~2025 소득구간 다층 분포")
print(raw.loc[raw["조사년도"].isin([2024, 2025]), "소득구간_다층"].value_counts(dropna=False))


## 3. 선호도 순위 테이블 생성

- 향후 희망 여가활동 1~3순위를 사용함.
- 1.5, 1.25, 1.0 순위 가중치를 적용함.
- 응답자 내부 정규화 후 최종가중치를 반영함.


In [ ]:
excluded_categories = ["분류범위외", "교통수단", "여행사", "음악", "체육용품"]
mapping["use_target_final"] = (
    mapping["use_target"].astype(bool)
    & ~mapping["category"].isin(excluded_categories)
)

rank_cols = {
    1: "향후 희망하는 여가활동 1순위",
    2: "향후 희망하는 여가활동 2순위",
    3: "향후 희망하는 여가활동 3순위",
}
rank_score = {1: 1.5, 2: 1.25, 3: 1.0}

preference_raw = raw.loc[
    raw["조사년도"].isin([2024, 2025]),
    [
        "응답자_ID", "최종가중치", "성별", "연령", "조사년도",
        "소득구간_50이진", "소득구간_다층"
    ] + list(rank_cols.values())
].copy()

preference_raw["성별_연령"] = (
    preference_raw["성별"].astype("Int64").astype(str)
    + "_"
    + preference_raw["연령"].astype("Int64").astype(str)
)

code_to_category = mapping.set_index("activity_code")["category"].to_dict()
code_to_use = mapping.set_index("activity_code")["use_target_final"].to_dict()

wide_records = []
long_records = []

for _, row in preference_raw.iterrows():
    valid_rows = []
    seen_categories = set()

    for rank_no, col in rank_cols.items():
        activity_code = row[col]
        if pd.isna(activity_code):
            continue

        activity_code = int(activity_code)
        category = code_to_category.get(activity_code)
        use_target = bool(code_to_use.get(activity_code, False))

        if not use_target:
            continue

        if category in seen_categories:
            continue

        seen_categories.add(category)
        valid_rows.append({
            "rank_no": rank_no,
            "rank_score": rank_score[rank_no],
            "target_category": category,
        })

    score_sum = sum(x["rank_score"] for x in valid_rows)

    wide_row = {
        "응답자_ID": row["응답자_ID"],
        "성별": row["성별"],
        "연령": row["연령"],
        "조사년도": row["조사년도"],
        "성별_연령": row["성별_연령"],
        "소득구간_50이진": row["소득구간_50이진"],
        "소득구간_다층": row["소득구간_다층"],
        "최종가중치": row["최종가중치"],
        "선호_유효순위수": len(valid_rows),
    }

    for i, valid in enumerate(valid_rows, start=1):
        wide_row[f"선호_유효중분류_{i}순위"] = valid["target_category"]
        long_records.append({
            "응답자_ID": row["응답자_ID"],
            "성별": row["성별"],
            "연령": row["연령"],
            "조사년도": row["조사년도"],
            "성별_연령": row["성별_연령"],
            "소득구간_50이진": row["소득구간_50이진"],
            "소득구간_다층": row["소득구간_다층"],
            "rank_no": valid["rank_no"],
            "rank_score": valid["rank_score"],
            "target_category": valid["target_category"],
            "sample_weight": row["최종가중치"] * valid["rank_score"] / score_sum if score_sum > 0 else 0,
        })

    wide_records.append(wide_row)

rank_base = pd.DataFrame(wide_records)
rank_long = pd.DataFrame(long_records)

print("응답자 테이블:", rank_base.shape)
print("순위 long 테이블:", rank_long.shape)
print(rank_base["선호_유효순위수"].value_counts().sort_index())


## 4. Split 설계

- train/test = 8:2
- train 내부 train/valid = 8:2
- shuffle + stratify 적용


In [ ]:
primary_target = (
    rank_long.sort_values(["응답자_ID", "rank_no"])
    .groupby("응답자_ID", as_index=False)["target_category"]
    .first()
    .rename(columns={"target_category": "primary_target"})
)

model_base = (
    rank_base.loc[rank_base["선호_유효순위수"] > 0]
    .merge(primary_target, on="응답자_ID", how="left")
    .copy()
)

def choose_strata(df, min_count=2):
    year_target = df["조사년도"].astype(str) + "_" + df["primary_target"].astype(str)
    if year_target.value_counts().min() >= min_count:
        return year_target

    target_only = df["primary_target"].astype(str)
    if target_only.value_counts().min() >= min_count:
        return target_only

    return None

train_valid_base, test_base = train_test_split(
    model_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(model_base, 2),
)

train_base, valid_base = train_test_split(
    train_valid_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(train_valid_base, 2),
)

train_data = rank_long[rank_long["응답자_ID"].isin(train_base["응답자_ID"])].copy()
valid_data = rank_long[rank_long["응답자_ID"].isin(valid_base["응답자_ID"])].copy()
test_data = rank_long[rank_long["응답자_ID"].isin(test_base["응답자_ID"])].copy()

classes = np.array(sorted(rank_long["target_category"].unique()))

split_summary = pd.DataFrame({
    "dataset": ["train", "valid", "test"],
    "respondents": [
        train_base["응답자_ID"].nunique(),
        valid_base["응답자_ID"].nunique(),
        test_base["응답자_ID"].nunique(),
    ],
})
split_summary["share"] = split_summary["respondents"] / model_base["응답자_ID"].nunique()
display(split_summary)


## 5. 모델 정의

- 비교 모델은 Multinomial Logistic, HistGradientBoosting으로 제한함.
- 소득구간 이진/다층을 각각 별도 실험으로 비교함.


In [ ]:
def make_models(feature_cols):
    onehot = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), feature_cols)],
        remainder="drop",
    )

    return {
        "multinomial_logistic": Pipeline([
            ("preprocess", onehot),
            ("model", LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0))
        ]),
        "hist_gradient_boosting": Pipeline([
            ("preprocess", onehot),
            ("model", HistGradientBoostingClassifier(
                max_iter=60,
                learning_rate=0.06,
                max_leaf_nodes=8,
                l2_regularization=1.0,
                random_state=42,
            ))
        ]),
    }

def fit_model(model, data, feature_cols):
    x = data[feature_cols].astype(str)
    y = data["target_category"]
    w = data["sample_weight"]
    return model.fit(x, y, model__sample_weight=w)

def align_proba(model, proba):
    model_classes = model.named_steps["model"].classes_
    aligned = np.zeros((proba.shape[0], len(classes)))
    class_to_idx = {c: i for i, c in enumerate(model_classes)}

    for j, c in enumerate(classes):
        if c in class_to_idx:
            aligned[:, j] = proba[:, class_to_idx[c]]

    row_sum = aligned.sum(axis=1, keepdims=True)
    return np.divide(aligned, row_sum, out=np.zeros_like(aligned), where=row_sum > 0)

def weighted_brier(y_true, proba, sample_weight):
    y_index = pd.Categorical(y_true, categories=classes).codes
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_index)), y_index] = 1
    return np.average(((proba - y_onehot) ** 2).sum(axis=1), weights=sample_weight)

def evaluate(experiment, model_name, model, data, dataset_name, feature_cols):
    x = data[feature_cols].astype(str)
    y = data["target_category"].to_numpy()
    w = data["sample_weight"].to_numpy()
    proba = align_proba(model, model.predict_proba(x))
    y_pred = classes[np.argmax(proba, axis=1)]

    return {
        "experiment": experiment,
        "model": model_name,
        "dataset": dataset_name,
        "LogLoss": log_loss(y, proba, labels=classes, sample_weight=w),
        "Top1_Accuracy": accuracy_score(y, y_pred, sample_weight=w),
        "Top3_HitRate": top_k_accuracy_score(y, proba, k=3, labels=classes, sample_weight=w),
        "Macro_F1": f1_score(y, y_pred, labels=classes, average="macro", sample_weight=w, zero_division=0),
        "Weighted_F1": f1_score(y, y_pred, labels=classes, average="weighted", sample_weight=w, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, y_pred, sample_weight=w),
        "Brier": weighted_brier(y, proba, w),
    }


## 6. 모델별 성능 실험

- base: 성별, 연령, 조사년도, 성별_연령
- income_binary: base + 소득구간_50이진
- income_band: base + 소득구간_다층


In [ ]:
experiments = {
    "base_sex_age": ["성별", "연령", "조사년도", "성별_연령"],
    "income_binary_50": ["성별", "연령", "조사년도", "성별_연령", "소득구간_50이진"],
    "income_band": ["성별", "연령", "조사년도", "성별_연령", "소득구간_다층"],
}

performance_rows = []
cv_rows = []

cv_base = train_valid_base.reset_index(drop=True)
cv_strata = choose_strata(cv_base, min_count=3)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for exp_name, feature_cols in experiments.items():
    for fold_no, (tr_idx, va_idx) in enumerate(skf.split(cv_base, cv_strata), start=1):
        tr_ids = cv_base.iloc[tr_idx]["응답자_ID"]
        va_ids = cv_base.iloc[va_idx]["응답자_ID"]

        fold_train_data = rank_long[rank_long["응답자_ID"].isin(tr_ids)].copy()
        fold_valid_data = rank_long[rank_long["응답자_ID"].isin(va_ids)].copy()

        for model_name, model in make_models(feature_cols).items():
            print("cv", exp_name, fold_no, model_name)
            model = fit_model(model, fold_train_data, feature_cols)
            row = evaluate(exp_name, model_name, model, fold_valid_data, "cv_valid", feature_cols)
            row["fold"] = fold_no
            cv_rows.append(row)

    for model_name, model in make_models(feature_cols).items():
        print("holdout", exp_name, model_name)
        model = fit_model(model, train_data, feature_cols)

        for dataset_name, dataset in [
            ("train", train_data),
            ("valid", valid_data),
            ("test", test_data),
        ]:
            performance_rows.append(
                evaluate(exp_name, model_name, model, dataset, dataset_name, feature_cols)
            )

cv_performance = pd.DataFrame(cv_rows)
performance = pd.DataFrame(performance_rows)

cv_summary = cv_performance.groupby(["experiment", "model"], as_index=False).agg(
    LogLoss_mean=("LogLoss", "mean"),
    LogLoss_std=("LogLoss", "std"),
    Top1_Accuracy_mean=("Top1_Accuracy", "mean"),
    Top3_HitRate_mean=("Top3_HitRate", "mean"),
    Macro_F1_mean=("Macro_F1", "mean"),
    Balanced_Accuracy_mean=("Balanced_Accuracy", "mean"),
    Brier_mean=("Brier", "mean"),
)

print("3-Fold CV")
display(cv_summary.sort_values("LogLoss_mean"))

print("Holdout Test")
display(performance[performance["dataset"] == "test"].sort_values("LogLoss"))


## 7. 시도·장애여부 추가 실험

- 기존 최적 후보였던 소득 다층 피처에 17개 시도와 장애여부를 추가함.
- 모델은 다항 로지스틱, HistGradientBoosting 두 개만 비교함.
- 타깃, 순위 가중치, train/valid/test 분할 방식은 이전 실험과 동일하게 유지함.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

BASE_PATH = Path.cwd()
while BASE_PATH.name != "oracle_mnc_project" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

RAW_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "source" / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"
MAPPING_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "processed" / "satisfaction" / "ml_activity_category_mapping.csv"

raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")
mapping.columns = ["activity_code", "activity_name", "category", "use_target"]

# 기준 중위소득 100%, 원/월
median_income_100 = {
    2024: {
        1: 2228445, 2: 3682609, 3: 4714657, 4: 5729913,
        5: 6695735, 6: 7618369, 7: 8514994,
    },
    2025: {
        1: 2392013, 2: 3932658, 3: 5025353, 4: 6097773,
        5: 7108192, 6: 8064805, 7: 8988428,
    },
}

income_mid_10k = {
    1: 50,
    2: 150,
    3: 250,
    4: 350,
    5: 450,
    6: 550,
    7: 650,
}

def household_size_cap(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x < 1:
        return np.nan
    return min(x, 7)

def median_ratio(row):
    year = int(row["조사년도"])
    size = household_size_cap(row["동거가구원수_생성"])
    income_code = row["가구소득_생성"]

    if pd.isna(size) or pd.isna(income_code):
        return np.nan

    income_won = income_mid_10k.get(int(income_code), np.nan) * 10000
    median_won = median_income_100.get(year, {}).get(size, np.nan)

    if pd.isna(income_won) or pd.isna(median_won) or median_won == 0:
        return np.nan

    return income_won / median_won

raw["중위소득비율_근사"] = raw.apply(median_ratio, axis=1)
raw["소득구간_다층"] = pd.cut(
    raw["중위소득비율_근사"],
    bins=[-np.inf, 0.5, 1.0, 1.5, np.inf],
    labels=["50이하", "50_100", "100_150", "150초과"],
).astype(str)
raw.loc[raw["중위소득비율_근사"].isna(), "소득구간_다층"] = "unknown"

excluded_categories = ["분류범위외", "교통수단", "여행사", "음악", "체육용품"]
mapping["use_target_final"] = (
    mapping["use_target"].astype(bool)
    & ~mapping["category"].isin(excluded_categories)
)

rank_cols = {
    1: "향후 희망하는 여가활동 1순위",
    2: "향후 희망하는 여가활동 2순위",
    3: "향후 희망하는 여가활동 3순위",
}
rank_score = {1: 1.5, 2: 1.25, 3: 1.0}

feature_cols = [
    "성별",
    "연령",
    "조사년도",
    "성별_연령",
    "소득구간_다층",
    "장애여부",
    "17개 시도",
]

preference_raw = raw.loc[
    raw["조사년도"].isin([2024, 2025]),
    [
        "응답자_ID", "최종가중치", "성별", "연령", "조사년도",
        "소득구간_다층", "장애여부", "17개 시도"
    ] + list(rank_cols.values())
].copy()

preference_raw["성별_연령"] = (
    preference_raw["성별"].astype("Int64").astype(str)
    + "_"
    + preference_raw["연령"].astype("Int64").astype(str)
)

for col in feature_cols:
    preference_raw[col] = preference_raw[col].astype("string").fillna("unknown")

code_to_category = mapping.set_index("activity_code")["category"].to_dict()
code_to_use = mapping.set_index("activity_code")["use_target_final"].to_dict()

wide_records = []
long_records = []

for _, row in preference_raw.iterrows():
    valid_rows = []
    seen_categories = set()

    for rank_no, col in rank_cols.items():
        activity_code = row[col]
        if pd.isna(activity_code):
            continue

        activity_code = int(activity_code)
        category = code_to_category.get(activity_code)
        use_target = bool(code_to_use.get(activity_code, False))

        if not use_target:
            continue

        if category in seen_categories:
            continue

        seen_categories.add(category)
        valid_rows.append({
            "rank_no": rank_no,
            "rank_score": rank_score[rank_no],
            "target_category": category,
        })

    score_sum = sum(x["rank_score"] for x in valid_rows)

    wide_row = {
        "응답자_ID": row["응답자_ID"],
        "성별": row["성별"],
        "연령": row["연령"],
        "조사년도": row["조사년도"],
        "성별_연령": row["성별_연령"],
        "소득구간_다층": row["소득구간_다층"],
        "장애여부": row["장애여부"],
        "17개 시도": row["17개 시도"],
        "최종가중치": row["최종가중치"],
        "선호_유효순위수": len(valid_rows),
    }

    for i, valid in enumerate(valid_rows, start=1):
        wide_row[f"선호_유효중분류_{i}순위"] = valid["target_category"]
        long_records.append({
            "응답자_ID": row["응답자_ID"],
            "성별": row["성별"],
            "연령": row["연령"],
            "조사년도": row["조사년도"],
            "성별_연령": row["성별_연령"],
            "소득구간_다층": row["소득구간_다층"],
            "장애여부": row["장애여부"],
            "17개 시도": row["17개 시도"],
            "rank_no": valid["rank_no"],
            "rank_score": valid["rank_score"],
            "target_category": valid["target_category"],
            "sample_weight": row["최종가중치"] * valid["rank_score"] / score_sum,
        })

    wide_records.append(wide_row)

rank_base_exp = pd.DataFrame(wide_records)
rank_long_exp = pd.DataFrame(long_records)

print("입력 피처")
print(feature_cols)

print("\n장애여부 분포")
print(rank_base_exp["장애여부"].value_counts(dropna=False))

print("\n17개 시도 분포 상위 10")
print(rank_base_exp["17개 시도"].value_counts(dropna=False).head(10))

print("\n유효 순위 수")
print(rank_base_exp["선호_유효순위수"].value_counts().sort_index())


In [ ]:
primary_target = (
    rank_long_exp.sort_values(["응답자_ID", "rank_no"])
    .groupby("응답자_ID", as_index=False)["target_category"]
    .first()
    .rename(columns={"target_category": "primary_target"})
)

model_base_exp = (
    rank_base_exp.loc[rank_base_exp["선호_유효순위수"] > 0]
    .merge(primary_target, on="응답자_ID", how="left")
    .copy()
)

def choose_strata(df, min_count=2):
    year_target = df["조사년도"].astype(str) + "_" + df["primary_target"].astype(str)
    if year_target.value_counts().min() >= min_count:
        return year_target

    target_only = df["primary_target"].astype(str)
    if target_only.value_counts().min() >= min_count:
        return target_only

    return None

train_valid_base_exp, test_base_exp = train_test_split(
    model_base_exp,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(model_base_exp, 2),
)

train_base_exp, valid_base_exp = train_test_split(
    train_valid_base_exp,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(train_valid_base_exp, 2),
)

train_data_exp = rank_long_exp[rank_long_exp["응답자_ID"].isin(train_base_exp["응답자_ID"])].copy()
valid_data_exp = rank_long_exp[rank_long_exp["응답자_ID"].isin(valid_base_exp["응답자_ID"])].copy()
test_data_exp = rank_long_exp[rank_long_exp["응답자_ID"].isin(test_base_exp["응답자_ID"])].copy()

classes_exp = np.array(sorted(rank_long_exp["target_category"].unique()))

def make_models_exp(feature_cols):
    onehot = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), feature_cols)],
        remainder="drop",
    )

    return {
        "multinomial_logistic": Pipeline([
            ("preprocess", onehot),
            ("model", LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0))
        ]),
        "hist_gradient_boosting": Pipeline([
            ("preprocess", onehot),
            ("model", HistGradientBoostingClassifier(
                max_iter=60,
                learning_rate=0.06,
                max_leaf_nodes=8,
                l2_regularization=1.0,
                random_state=42,
            ))
        ]),
    }

def fit_model_exp(model, data, feature_cols):
    x = data[feature_cols].astype(str)
    y = data["target_category"]
    w = data["sample_weight"]
    return model.fit(x, y, model__sample_weight=w)

def align_proba_exp(model, proba):
    model_classes = model.named_steps["model"].classes_
    aligned = np.zeros((proba.shape[0], len(classes_exp)))
    class_to_idx = {c: i for i, c in enumerate(model_classes)}

    for j, c in enumerate(classes_exp):
        if c in class_to_idx:
            aligned[:, j] = proba[:, class_to_idx[c]]

    row_sum = aligned.sum(axis=1, keepdims=True)
    return np.divide(aligned, row_sum, out=np.zeros_like(aligned), where=row_sum > 0)

def weighted_brier_exp(y_true, proba, sample_weight):
    y_index = pd.Categorical(y_true, categories=classes_exp).codes
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_index)), y_index] = 1
    return np.average(((proba - y_onehot) ** 2).sum(axis=1), weights=sample_weight)

def evaluate_exp(experiment, model_name, model, data, dataset_name, feature_cols):
    x = data[feature_cols].astype(str)
    y = data["target_category"].to_numpy()
    w = data["sample_weight"].to_numpy()
    proba = align_proba_exp(model, model.predict_proba(x))
    y_pred = classes_exp[np.argmax(proba, axis=1)]

    return {
        "experiment": experiment,
        "model": model_name,
        "dataset": dataset_name,
        "LogLoss": log_loss(y, proba, labels=classes_exp, sample_weight=w),
        "Top1_Accuracy": accuracy_score(y, y_pred, sample_weight=w),
        "Top3_HitRate": top_k_accuracy_score(y, proba, k=3, labels=classes_exp, sample_weight=w),
        "Macro_F1": f1_score(y, y_pred, labels=classes_exp, average="macro", sample_weight=w, zero_division=0),
        "Weighted_F1": f1_score(y, y_pred, labels=classes_exp, average="weighted", sample_weight=w, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, y_pred),
        "Brier": weighted_brier_exp(y, proba, w),
    }

experiment_name = "income_band_sido_disability"

performance_rows = []
cv_rows = []

cv_base_exp = train_valid_base_exp.reset_index(drop=True)
cv_strata_exp = choose_strata(cv_base_exp, min_count=3)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for fold_no, (tr_idx, va_idx) in enumerate(skf.split(cv_base_exp, cv_strata_exp), start=1):
    tr_ids = cv_base_exp.iloc[tr_idx]["응답자_ID"]
    va_ids = cv_base_exp.iloc[va_idx]["응답자_ID"]

    fold_train_data = rank_long_exp[rank_long_exp["응답자_ID"].isin(tr_ids)].copy()
    fold_valid_data = rank_long_exp[rank_long_exp["응답자_ID"].isin(va_ids)].copy()

    for model_name, model in make_models_exp(feature_cols).items():
        print("cv", experiment_name, fold_no, model_name)
        model = fit_model_exp(model, fold_train_data, feature_cols)
        row = evaluate_exp(experiment_name, model_name, model, fold_valid_data, "cv_valid", feature_cols)
        row["fold"] = fold_no
        cv_rows.append(row)

for model_name, model in make_models_exp(feature_cols).items():
    print("holdout", experiment_name, model_name)
    model = fit_model_exp(model, train_data_exp, feature_cols)

    for dataset_name, dataset in [
        ("train", train_data_exp),
        ("valid", valid_data_exp),
        ("test", test_data_exp),
    ]:
        performance_rows.append(
            evaluate_exp(experiment_name, model_name, model, dataset, dataset_name, feature_cols)
        )

cv_performance_added = pd.DataFrame(cv_rows)
performance_added = pd.DataFrame(performance_rows)

cv_summary_added = cv_performance_added.groupby(["experiment", "model"], as_index=False).agg(
    LogLoss_mean=("LogLoss", "mean"),
    LogLoss_std=("LogLoss", "std"),
    Top1_Accuracy_mean=("Top1_Accuracy", "mean"),
    Top3_HitRate_mean=("Top3_HitRate", "mean"),
    Macro_F1_mean=("Macro_F1", "mean"),
    Balanced_Accuracy_mean=("Balanced_Accuracy", "mean"),
    Brier_mean=("Brier", "mean"),
)

print("3-Fold CV")
display(cv_summary_added.sort_values("LogLoss_mean"))

print("Holdout Test")
display(performance_added[performance_added["dataset"] == "test"].sort_values("LogLoss"))


## 8. 가계동향조사 문화소비 파생변수 추가 실험

- 가계동향조사에서 연도·소득구간·성별·연령대별 문화소비 평균을 생성함.
- 여가조사 응답자의 조사년도, 가구소득, 성별, 연령대에 맞춰 문화소비 파생변수를 결합함.
- 기존 후보와 문화소비 추가 후보를 같은 분할 조건에서 비교함.


In [ ]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

os.environ["LOKY_MAX_CPU_COUNT"] = "4"
warnings.filterwarnings("ignore")

BASE_PATH = Path.cwd()
while BASE_PATH.name != "oracle_mnc_project" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

RAW_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "source" / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"
MAPPING_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "processed" / "satisfaction" / "ml_activity_category_mapping.csv"
ECONOMY_PATH = next((BASE_PATH / "data" / "raw" / "economy").iterdir())

raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")
mapping.columns = ["activity_code", "activity_name", "category", "use_target"]

print("가계동향조사 경로:", ECONOMY_PATH)


### 8-1. 가계동향조사 문화소비 파생변수 생성

- 가계동향조사는 가구 단위 지출자료이므로 개인 ID로 직접 결합하지 않음.
- 가구원을 성별·연령대로 펼친 뒤, 집단별 평균 문화소비 특성을 lookup 변수로 생성함.
- 결합 기준: 연도, 소득구간, 성별, 연령대.


In [ ]:
economy_base_cols = [
    "조사연월",
    "가구원수",
    "가중값",
    "소득구간코드",
    "가계지출_소비지출금액",
    "가계지출_소비지출_오락문화비",
    "가계지출_소비지출_오락문화_문화서비스이용금액",
    "가계지출_소비지출_오락문화_운동오락서비스이용금액",
    "가계지출_소비지출_오락문화_서적구입비",
    "가계지출_소비지출_오락문화_단체여행경비",
]

economy_member_cols = []
for i in range(1, 10):
    prefix = "가구주" if i == 1 else f"가구원{i}"
    economy_member_cols.extend([f"{prefix}_성별코드", f"{prefix}_연령"])

economy_frames = []
for file_path in sorted(ECONOMY_PATH.glob("*.csv")):
    cols = pd.read_csv(file_path, encoding="cp949", nrows=0).columns.tolist()
    usecols = [col for col in economy_base_cols + economy_member_cols if col in cols]
    temp = pd.read_csv(file_path, encoding="cp949", usecols=usecols)
    temp["연도"] = temp["조사연월"].astype(str).str[:4].astype(int)
    economy_frames.append(temp)

economy = pd.concat(economy_frames, ignore_index=True)
economy = economy[economy["연도"].isin([2024, 2025])].copy()

for col in economy_base_cols:
    if col in economy.columns and col != "조사연월":
        economy[col] = pd.to_numeric(economy[col], errors="coerce")

# 가계동향 7=600~700만원, 8=700만원 이상 / 여가조사 7=600만원 이상
economy["여가조사_소득코드"] = economy["소득구간코드"].clip(upper=7)

amount_cols = [
    "가계지출_소비지출_오락문화비",
    "가계지출_소비지출_오락문화_문화서비스이용금액",
    "가계지출_소비지출_오락문화_운동오락서비스이용금액",
    "가계지출_소비지출_오락문화_서적구입비",
    "가계지출_소비지출_오락문화_단체여행경비",
]

for col in amount_cols:
    economy[f"{col}_1인당"] = np.where(
        economy["가구원수"] > 0,
        economy[col] / economy["가구원수"],
        np.nan,
    )

economy["문화여가지출비중"] = np.where(
    economy["가계지출_소비지출금액"] > 0,
    economy["가계지출_소비지출_오락문화비"] / economy["가계지출_소비지출금액"],
    np.nan,
)
economy["문화서비스비중_오락문화내"] = np.where(
    economy["가계지출_소비지출_오락문화비"] > 0,
    economy["가계지출_소비지출_오락문화_문화서비스이용금액"] / economy["가계지출_소비지출_오락문화비"],
    np.nan,
)
economy["운동오락서비스비중_오락문화내"] = np.where(
    economy["가계지출_소비지출_오락문화비"] > 0,
    economy["가계지출_소비지출_오락문화_운동오락서비스이용금액"] / economy["가계지출_소비지출_오락문화비"],
    np.nan,
)

person_frames = []
person_value_cols = [
    "연도",
    "여가조사_소득코드",
    "가중값",
    "가계지출_소비지출_오락문화비_1인당",
    "가계지출_소비지출_오락문화_문화서비스이용금액_1인당",
    "가계지출_소비지출_오락문화_운동오락서비스이용금액_1인당",
    "가계지출_소비지출_오락문화_서적구입비_1인당",
    "가계지출_소비지출_오락문화_단체여행경비_1인당",
    "문화여가지출비중",
    "문화서비스비중_오락문화내",
    "운동오락서비스비중_오락문화내",
]

for i in range(1, 10):
    prefix = "가구주" if i == 1 else f"가구원{i}"
    sex_col = f"{prefix}_성별코드"
    age_col = f"{prefix}_연령"

    if sex_col not in economy.columns or age_col not in economy.columns:
        continue

    temp = economy[person_value_cols + [sex_col, age_col]].copy()
    temp = temp.rename(columns={sex_col: "성별", age_col: "연령값"})
    person_frames.append(temp)

economy_person = pd.concat(person_frames, ignore_index=True)
economy_person["성별"] = pd.to_numeric(economy_person["성별"], errors="coerce")
economy_person["연령값"] = pd.to_numeric(economy_person["연령값"], errors="coerce")
economy_person = economy_person[
    economy_person["성별"].isin([1, 2])
    & economy_person["연령값"].notna()
    & (economy_person["연령값"] >= 15)
].copy()

economy_person["연령"] = pd.cut(
    economy_person["연령값"],
    bins=[14, 19, 29, 39, 49, 59, 69, np.inf],
    labels=[1, 2, 3, 4, 5, 6, 7],
).astype(int)

def weighted_average(group, value_col):
    value = pd.to_numeric(group[value_col], errors="coerce")
    weight = pd.to_numeric(group["가중값"], errors="coerce")
    ok = value.notna() & weight.notna() & (weight > 0)
    if ok.sum() == 0:
        return np.nan
    return np.average(value[ok], weights=weight[ok])

join_keys = ["연도", "여가조사_소득코드", "성별", "연령"]
economy_feature_cols = [
    "가계지출_소비지출_오락문화비_1인당",
    "가계지출_소비지출_오락문화_문화서비스이용금액_1인당",
    "가계지출_소비지출_오락문화_운동오락서비스이용금액_1인당",
    "가계지출_소비지출_오락문화_서적구입비_1인당",
    "가계지출_소비지출_오락문화_단체여행경비_1인당",
    "문화여가지출비중",
    "문화서비스비중_오락문화내",
    "운동오락서비스비중_오락문화내",
]

lookup = economy_person.groupby(join_keys, dropna=False).size().reset_index(name="eco_집계표본수")

for col in economy_feature_cols:
    value = (
        economy_person
        .groupby(join_keys, dropna=False)
        .apply(lambda x, c=col: weighted_average(x, c))
        .reset_index(name=f"eco_{col}")
    )
    lookup = lookup.merge(value, on=join_keys, how="left")

lookup = lookup.rename(columns={"연도": "조사년도"})

print("가계동향 원자료 구조:", economy.shape)
print("가계동향 15세 이상 가구원 구조:", economy_person.shape)
print("가계동향 lookup 구조:", lookup.shape)
print("lookup 최소 표본수:", lookup["eco_집계표본수"].min())
print("lookup 표본수 30 미만:", (lookup["eco_집계표본수"] < 30).sum())
display(lookup.head())


### 8-2. 여가조사 학습테이블 결합

- 가계동향 소득구간 7·8번은 여가조사의 7번 구간으로 통합함.
- 결합 후 결측 파생변수는 전체 lookup 중앙값으로 보정함.
- 타깃은 향후 희망 여가활동 1~3순위의 유효 중분류를 사용함.


In [ ]:
# 기준 중위소득 100%, 원/월
median_income_100 = {
    2024: {
        1: 2228445, 2: 3682609, 3: 4714657, 4: 5729913,
        5: 6695735, 6: 7618369, 7: 8514994,
    },
    2025: {
        1: 2392013, 2: 3932658, 3: 5025353, 4: 6097773,
        5: 7108192, 6: 8064805, 7: 8988428,
    },
}

income_mid_10k = {
    1: 50,
    2: 150,
    3: 250,
    4: 350,
    5: 450,
    6: 550,
    7: 650,
}

def household_size_cap(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x < 1:
        return np.nan
    return min(x, 7)

def median_ratio(row):
    year = int(row["조사년도"])
    size = household_size_cap(row["동거가구원수_생성"])
    income_code = row["가구소득_생성"]

    if pd.isna(size) or pd.isna(income_code):
        return np.nan

    income_won = income_mid_10k.get(int(income_code), np.nan) * 10000
    median_won = median_income_100.get(year, {}).get(size, np.nan)

    if pd.isna(income_won) or pd.isna(median_won) or median_won == 0:
        return np.nan

    return income_won / median_won

raw["중위소득비율_근사"] = raw.apply(median_ratio, axis=1)
raw["소득구간_다층"] = pd.cut(
    raw["중위소득비율_근사"],
    bins=[-np.inf, 0.5, 1.0, 1.5, np.inf],
    labels=["50이하", "50_100", "100_150", "150초과"],
).astype(str)
raw.loc[raw["중위소득비율_근사"].isna(), "소득구간_다층"] = "unknown"
raw["여가조사_소득코드"] = pd.to_numeric(raw["가구소득_생성"], errors="coerce")

excluded_categories = ["분류범위외", "교통수단", "여행사", "음악", "체육용품"]
mapping["use_target_final"] = (
    mapping["use_target"].astype(bool)
    & ~mapping["category"].isin(excluded_categories)
)

rank_cols = {
    1: "향후 희망하는 여가활동 1순위",
    2: "향후 희망하는 여가활동 2순위",
    3: "향후 희망하는 여가활동 3순위",
}
rank_score = {1: 1.5, 2: 1.25, 3: 1.0}

base_feature_cols = [
    "성별",
    "연령",
    "조사년도",
    "성별_연령",
    "소득구간_다층",
    "장애여부",
    "17개 시도",
]

numeric_feature_cols = [col for col in lookup.columns if col.startswith("eco_") and col != "eco_집계표본수"]
numeric_feature_cols = numeric_feature_cols + ["eco_집계표본수"]

preference_raw = raw.loc[
    raw["조사년도"].isin([2024, 2025]),
    [
        "응답자_ID", "최종가중치", "성별", "연령", "조사년도",
        "소득구간_다층", "여가조사_소득코드", "장애여부", "17개 시도"
    ] + list(rank_cols.values())
].copy()

preference_raw["성별_연령"] = (
    preference_raw["성별"].astype("Int64").astype(str)
    + "_"
    + preference_raw["연령"].astype("Int64").astype(str)
)

preference_raw = preference_raw.merge(
    lookup,
    on=["조사년도", "여가조사_소득코드", "성별", "연령"],
    how="left",
)

missing_before = preference_raw[numeric_feature_cols].isna().sum()
for col in numeric_feature_cols:
    preference_raw[col] = preference_raw[col].fillna(lookup[col].median())

for col in base_feature_cols:
    preference_raw[col] = preference_raw[col].astype("string").fillna("unknown")

print("가계동향 변수 결합 후 구조:", preference_raw.shape)
print("가계동향 numeric feature 결측 보정 전")
print(missing_before.to_string())

code_to_category = mapping.set_index("activity_code")["category"].to_dict()
code_to_use = mapping.set_index("activity_code")["use_target_final"].to_dict()

wide_records = []
long_records = []

for _, row in preference_raw.iterrows():
    valid_rows = []
    seen_categories = set()

    for rank_no, col in rank_cols.items():
        activity_code = row[col]
        if pd.isna(activity_code):
            continue

        activity_code = int(activity_code)
        category = code_to_category.get(activity_code)
        use_target = bool(code_to_use.get(activity_code, False))

        if not use_target:
            continue

        if category in seen_categories:
            continue

        seen_categories.add(category)
        valid_rows.append({
            "rank_no": rank_no,
            "rank_score": rank_score[rank_no],
            "target_category": category,
        })

    score_sum = sum(x["rank_score"] for x in valid_rows)

    wide_row = {
        "응답자_ID": row["응답자_ID"],
        "성별": row["성별"],
        "연령": row["연령"],
        "조사년도": row["조사년도"],
        "성별_연령": row["성별_연령"],
        "소득구간_다층": row["소득구간_다층"],
        "장애여부": row["장애여부"],
        "17개 시도": row["17개 시도"],
        "최종가중치": row["최종가중치"],
        "선호_유효순위수": len(valid_rows),
    }
    for col in numeric_feature_cols:
        wide_row[col] = row[col]

    for i, valid in enumerate(valid_rows, start=1):
        wide_row[f"선호_유효중분류_{i}순위"] = valid["target_category"]
        long_record = {
            "응답자_ID": row["응답자_ID"],
            "성별": row["성별"],
            "연령": row["연령"],
            "조사년도": row["조사년도"],
            "성별_연령": row["성별_연령"],
            "소득구간_다층": row["소득구간_다층"],
            "장애여부": row["장애여부"],
            "17개 시도": row["17개 시도"],
            "rank_no": valid["rank_no"],
            "rank_score": valid["rank_score"],
            "target_category": valid["target_category"],
            "sample_weight": row["최종가중치"] * valid["rank_score"] / score_sum,
        }
        for col in numeric_feature_cols:
            long_record[col] = row[col]
        long_records.append(long_record)

    wide_records.append(wide_row)

rank_base_eco = pd.DataFrame(wide_records)
rank_long_eco = pd.DataFrame(long_records)

print("응답자 테이블:", rank_base_eco.shape)
print("순위 long 테이블:", rank_long_eco.shape)
print("유효 순위 수")
print(rank_base_eco["선호_유효순위수"].value_counts().sort_index())


### 8-3. 모델 실험

- 비교 1: 소득다층 + 시도 + 장애여부
- 비교 2: 비교 1 + 가계동향 문화소비 파생변수
- 모델: 다항 로지스틱, HistGradientBoosting


In [ ]:
primary_target = (
    rank_long_eco.sort_values(["응답자_ID", "rank_no"])
    .groupby("응답자_ID", as_index=False)["target_category"]
    .first()
    .rename(columns={"target_category": "primary_target"})
)

model_base_eco = (
    rank_base_eco.loc[rank_base_eco["선호_유효순위수"] > 0]
    .merge(primary_target, on="응답자_ID", how="left")
    .copy()
)

def choose_strata(df, min_count=2):
    year_target = df["조사년도"].astype(str) + "_" + df["primary_target"].astype(str)
    if year_target.value_counts().min() >= min_count:
        return year_target

    target_only = df["primary_target"].astype(str)
    if target_only.value_counts().min() >= min_count:
        return target_only

    return None

train_valid_base_eco, test_base_eco = train_test_split(
    model_base_eco,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(model_base_eco, 2),
)

train_base_eco, valid_base_eco = train_test_split(
    train_valid_base_eco,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(train_valid_base_eco, 2),
)

train_data_eco = rank_long_eco[rank_long_eco["응답자_ID"].isin(train_base_eco["응답자_ID"])].copy()
valid_data_eco = rank_long_eco[rank_long_eco["응답자_ID"].isin(valid_base_eco["응답자_ID"])].copy()
test_data_eco = rank_long_eco[rank_long_eco["응답자_ID"].isin(test_base_eco["응답자_ID"])].copy()

classes_eco = np.array(sorted(rank_long_eco["target_category"].unique()))

def make_models_exp(cat_cols, num_cols):
    preprocess = ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", StandardScaler(), num_cols),
        ],
        remainder="drop",
    )

    return {
        "multinomial_logistic": Pipeline([
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0))
        ]),
        "hist_gradient_boosting": Pipeline([
            ("preprocess", preprocess),
            ("model", HistGradientBoostingClassifier(
                max_iter=60,
                learning_rate=0.06,
                max_leaf_nodes=8,
                l2_regularization=1.0,
                random_state=42,
            ))
        ]),
    }

def fit_model_exp(model, data, cat_cols, num_cols):
    x = data[cat_cols + num_cols].copy()
    x[cat_cols] = x[cat_cols].astype(str)
    y = data["target_category"]
    w = data["sample_weight"]
    return model.fit(x, y, model__sample_weight=w)

def align_proba_exp(model, proba):
    model_classes = model.named_steps["model"].classes_
    aligned = np.zeros((proba.shape[0], len(classes_eco)))
    class_to_idx = {c: i for i, c in enumerate(model_classes)}

    for j, c in enumerate(classes_eco):
        if c in class_to_idx:
            aligned[:, j] = proba[:, class_to_idx[c]]

    row_sum = aligned.sum(axis=1, keepdims=True)
    return np.divide(aligned, row_sum, out=np.zeros_like(aligned), where=row_sum > 0)

def weighted_brier_exp(y_true, proba, sample_weight):
    y_index = pd.Categorical(y_true, categories=classes_eco).codes
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_index)), y_index] = 1
    return np.average(((proba - y_onehot) ** 2).sum(axis=1), weights=sample_weight)

def evaluate_exp(experiment, model_name, model, data, dataset_name, cat_cols, num_cols):
    x = data[cat_cols + num_cols].copy()
    x[cat_cols] = x[cat_cols].astype(str)
    y = data["target_category"].to_numpy()
    w = data["sample_weight"].to_numpy()
    proba = align_proba_exp(model, model.predict_proba(x))
    y_pred = classes_eco[np.argmax(proba, axis=1)]

    return {
        "experiment": experiment,
        "model": model_name,
        "dataset": dataset_name,
        "LogLoss": log_loss(y, proba, labels=classes_eco, sample_weight=w),
        "Top1_Accuracy": accuracy_score(y, y_pred, sample_weight=w),
        "Top3_HitRate": top_k_accuracy_score(y, proba, k=3, labels=classes_eco, sample_weight=w),
        "Macro_F1": f1_score(y, y_pred, labels=classes_eco, average="macro", sample_weight=w, zero_division=0),
        "Weighted_F1": f1_score(y, y_pred, labels=classes_eco, average="weighted", sample_weight=w, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, y_pred),
        "Brier": weighted_brier_exp(y, proba, w),
    }

experiments = {
    "income_band_sido_disability": {
        "cat_cols": base_feature_cols,
        "num_cols": [],
    },
    "income_band_sido_disability_economy": {
        "cat_cols": base_feature_cols,
        "num_cols": numeric_feature_cols,
    },
}

performance_rows = []
cv_rows = []

cv_base_eco = train_valid_base_eco.reset_index(drop=True)
cv_strata_eco = choose_strata(cv_base_eco, min_count=3)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for exp_name, cols in experiments.items():
    cat_cols = cols["cat_cols"]
    num_cols = cols["num_cols"]

    for fold_no, (tr_idx, va_idx) in enumerate(skf.split(cv_base_eco, cv_strata_eco), start=1):
        tr_ids = cv_base_eco.iloc[tr_idx]["응답자_ID"]
        va_ids = cv_base_eco.iloc[va_idx]["응답자_ID"]

        fold_train_data = rank_long_eco[rank_long_eco["응답자_ID"].isin(tr_ids)].copy()
        fold_valid_data = rank_long_eco[rank_long_eco["응답자_ID"].isin(va_ids)].copy()

        for model_name, model in make_models_exp(cat_cols, num_cols).items():
            print("cv", exp_name, fold_no, model_name)
            model = fit_model_exp(model, fold_train_data, cat_cols, num_cols)
            row = evaluate_exp(exp_name, model_name, model, fold_valid_data, "cv_valid", cat_cols, num_cols)
            row["fold"] = fold_no
            cv_rows.append(row)

    for model_name, model in make_models_exp(cat_cols, num_cols).items():
        print("holdout", exp_name, model_name)
        model = fit_model_exp(model, train_data_eco, cat_cols, num_cols)

        for dataset_name, dataset in [
            ("train", train_data_eco),
            ("valid", valid_data_eco),
            ("test", test_data_eco),
        ]:
            performance_rows.append(
                evaluate_exp(exp_name, model_name, model, dataset, dataset_name, cat_cols, num_cols)
            )

cv_performance_eco = pd.DataFrame(cv_rows)
performance_eco = pd.DataFrame(performance_rows)

cv_summary_eco = cv_performance_eco.groupby(["experiment", "model"], as_index=False).agg(
    LogLoss_mean=("LogLoss", "mean"),
    LogLoss_std=("LogLoss", "std"),
    Top1_Accuracy_mean=("Top1_Accuracy", "mean"),
    Top3_HitRate_mean=("Top3_HitRate", "mean"),
    Macro_F1_mean=("Macro_F1", "mean"),
    Balanced_Accuracy_mean=("Balanced_Accuracy", "mean"),
    Brier_mean=("Brier", "mean"),
)

print("3-Fold CV")
display(cv_summary_eco.sort_values("LogLoss_mean"))

print("Holdout Test")
display(performance_eco[performance_eco["dataset"] == "test"].sort_values("LogLoss"))
